# 09. Naive Bayes (GaussianNB)

Assumes feature independence. Needs dense features (`dense=True`).
Tune `var_smoothing` with stratified CV (F1).

In [1]:
import sys
sys.path.append("..")

from sklearn.naive_bayes import GaussianNB

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult("train"))
X_test, y_test = split_features_target(load_adult("test"))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [2]:
candidates = [1e-11, 1e-9, 1e-7, 1e-5, 1e-3, 1e-1]
selection_metric = "f1"

cv_by_smooth = {}
best_smooth = None
best_score = -1.0

for smooth in candidates:
    model = GaussianNB(var_smoothing=smooth)
    pipe = make_model_pipeline(model, dense=True)
    cv = cross_val_scores(pipe, X, y)
    cv_by_smooth[smooth] = cv
    score = cv[selection_metric]["mean"]
    print(f"var_smoothing={smooth}  CV {selection_metric}={score:.4f}  (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_smooth = smooth

print("selected var_smoothing:", best_smooth)

var_smoothing=1e-11  CV f1=0.5364  (±0.0146)
var_smoothing=1e-09  CV f1=0.5465  (±0.0161)
var_smoothing=1e-07  CV f1=0.5615  (±0.0119)
var_smoothing=1e-05  CV f1=0.5798  (±0.0096)
var_smoothing=0.001  CV f1=0.6111  (±0.0041)
var_smoothing=0.1  CV f1=0.6234  (±0.0078)
selected var_smoothing: 0.1


In [3]:
final_model = GaussianNB(var_smoothing=best_smooth)

cv = cv_by_smooth[best_smooth]
pipe2 = make_model_pipeline(final_model, dense=True)
test = evaluate_on_test(pipe2, X, y, X_test, y_test)

path = save_results(
    "naive_bayes",
    cv,
    test,
    extra={
        "dense": True,
        "var_smoothing": best_smooth,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_var_smoothing": {
            str(s): {m: v["mean"] for m, v in summary.items()}
            for s, summary in cv_by_smooth.items()
        },
    },
)

print("Test:", test["metrics"])
print("Confusion matrix:", test["confusion_matrix"])
print("saved:", path)

Test: {'accuracy': 0.7511823598059088, 'balanced_accuracy': 0.7943700377689413, 'f1': 0.624594569548698, 'roc_auc': 0.8811168361491194, 'average_precision': 0.6985009204556353}
Confusion matrix: [[8860, 3575], [476, 3370]]
saved: /Users/kang/Desktop/ML project/Estimating-Income/results/naive_bayes.json
